In [3]:
# later replace with db
import json
with open('port-folio.json', 'r') as file:
    data = json.load(file)

In [10]:
import time

In [4]:
len(data)

10

In [5]:
data[0]

{'user_id': 'f47ac10b-58cc-4372-a567-0e02b2c3d479',
 'bio': 'Specialist in Large Language Model architecture and Agentic AI workflows. Experienced in fine-tuning models and building robust RAG pipelines.',
 'skills': ['Python',
  'PyTorch',
  'LLM Architecture',
  'RAG',
  'Agentic AI',
  'LoRA'],
 'hourly_rate': 85.0,
 'availability_status': 'Available (10-20 hrs/week)',
 'education': [{'institution': 'National Institute of Technology Warangal',
   'degree': 'B.Tech in Computer Science',
   'start_year': 2023,
   'end_year': 2027}],
 'work_experience': [{'company': 'AI Innovators Ltd',
   'role': 'Machine Learning Intern',
   'start_date': '2025-05-01',
   'end_date': '2025-07-31',
   'description': 'Implemented Direct Preference Optimization (DPO) on open-source models to improve response accuracy by 14%.'}],
 'projects': [{'title': 'Modular Storyboard Generation System',
   'description': 'An AI-driven pipeline converting natural language scripts into 2D animations using structured 

In [6]:
with open('projects.json','r') as file:
    projects=json.load(file)

In [8]:
project=projects[0]

In [9]:
project

{'project_id': 'proj_a1b2c3d4',
 'client_id': 'client_98765',
 'title': 'Database Normalization and PostgreSQL Migration',
 'description': 'We need a database architect to redesign our legacy inventory schema. The current structure has multiple BCNF violations causing update anomalies. You will be responsible for normalizing the tables to 3NF/BCNF while ensuring dependency preservation, and migrating the existing data to a new PostgreSQL instance.',
 'required_skills': ['PostgreSQL',
  'Database Normalization',
  'Data Migration',
  'SQL'],
 'budget_range': {'min': 1500, 'max': 3000},
 'deadline': '2026-06-30T23:59:59Z',
 'project_type': 'Fixed',
 'status': 'Open',
 'created_at': '2026-05-20T15:17:31Z'}

In [ ]:
#buget
if project['project_type']=='Fixed':
    weeks=project['deadline']-project['created_at']
else:
    pass

In [ ]:
import re
import os
from datetime import datetime, timezone
from typing import List, Dict, Any
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

# ==========================================
# 1. FREE API CONFIGURATION
# ==========================================
# Set your free API key in your environment variables:

# The Chat Model (For your Heavy Generator / Vagueness Checker)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7845.52it/s]


In [37]:
# NEW
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

In [40]:
import json
from langchain_core.prompts import PromptTemplate

def get_best_project_via_vector(project_desc: str, freelancer_projects: List[Dict]) -> Dict:
    if not freelancer_projects:
        return "No project found"

    # Convert freelancer projects into LangChain Documents
    docs = []
    for proj in freelancer_projects:
        content = f"Title: {proj.get('title', '')}\nDescription: {proj.get('description', '')}\nTech Stack: {', '.join(proj.get('tech_stack', []))}"
        docs.append(Document(page_content=content))

    # Build an ephemeral, in-memory FAISS vector database
    vectorstore = FAISS.from_documents(docs, embeddings)

    # Perform similarity search using cosine distance (normalized to 0-1 relevance)
    results = vectorstore.similarity_search(project_desc, k=1)

    if results and len(results) > 0:
        # results[0] is a tuple of (Document, Relevance_Score)
        # Ensure the score stays within bounds for your math equation
        #print(results)
        return results[0].page_content
    return "No relevant projects found"


def get_llm_relevance_score(client_project_desc: str, freelancer_project_text: str) -> float:
    
    # 2. Build the strict prompt (Notice we use freelancer_project_text directly now)
    prompt_template = PromptTemplate.from_template("""
    You are an expert technical recruiter evaluating a freelancer's portfolio project against a client's requirements.
    
    Client Requirements:
    {client}
    
    Freelancer's Past Project:
    {freelancer}
    
    Calculate a relevance score between 0.0 (completely irrelevant) and 1.0 (perfect match).
    You MUST respond with a valid JSON object in this exact format: {{"score": 0.85}}
    Do not include markdown blocks or any other text.
    """)

    prompt = prompt_template.format(client=client_project_desc, freelancer=freelancer_project_text)

    # 3. Call the LLM
    try:
        # Assuming `llm` is your ChatGoogleGenerativeAI instance from earlier
        response = llm.invoke(prompt)
        
        # Strip any markdown formatting the LLM might sneak in
        clean_json = response.content.replace("```json", "").replace("```", "").strip()
        
        # Parse the JSON and extract the float
        score_data = json.loads(clean_json)
        return float(score_data.get("score", 0.0))
        
    except Exception as e:
        print(f"LLM Scoring Failed: {e}")
        return 0.0 # Safe fallback

# ==========================================
# 3. CORE FILTER: BUDGET & AVAILABILITY
# ==========================================
def extract_weekly_hours(availability_str: str) -> int:
    if "Unavailable" in availability_str:
        return 0
    match = re.search(r'(\d+)', availability_str)
    return int(match.group(1)) if match else 0

def passes_hard_constraints(freelancer: Dict[str, Any], project: Dict[str, Any], current_date: datetime) -> bool:
    weekly_hours = extract_weekly_hours(freelancer.get("availability_status", ""))
    
    if weekly_hours == 0:
        return False
        
    freelancer_rate = freelancer.get("hourly_rate", 0)
    budget_range = project.get("budget_range", {"min": 0, "max": 0})
    
    if project.get("project_type") == "Hourly":
        max_acceptable_rate = budget_range["max"] * 1.2
        if freelancer_rate > max_acceptable_rate:
            return False
            
    elif project.get("project_type") == "Fixed":
        deadline_str = project.get("deadline")
        deadline = datetime.fromisoformat(deadline_str.replace("Z", "+00:00"))
        
        days_to_deadline = (deadline - current_date).days
        weeks_to_deadline = max(1, days_to_deadline / 7)
        
        max_capacity_cost = weeks_to_deadline * weekly_hours * freelancer_rate
        
        if max_capacity_cost < (budget_range["min"] * 0.5):
            return False
            
    return True

# ==========================================
# 4. SCORING ENGINE (From Whiteboard Math)
# ==========================================
def calculate_match_score(freelancer: Dict[str, Any], project: Dict[str, Any]) -> float:
    rating = freelancer.get("rating", 4.0) 
    active_days = freelancer.get("active_days_per_week", 5) 
    discription=project.get("description", "")
    # --- UPDATED AI SCORING LOGIC ---
    # Step 1: Use FAISS to instantly find the freelancer's single most relevant project (Fast)
    best_project_doc = get_best_project_via_vector(project.get("description", ""), freelancer.get("projects", []))
    
    # Step 2: Use the LLM to heavily analyze and score that specific project (Intelligent)
    if best_project_doc:
        llm_score = get_llm_relevance_score(discription, best_project_doc)
    else:
        llm_score = 0.0
        
    # Apply your whiteboard weight
    weighted_project = llm_score * 0.4
    # ---------------------------------
    
    required_skills = set(project.get("required_skills", []))
    freelancer_skills = set(freelancer.get("skills", []))
    
    if required_skills:
        skill_overlap = len(required_skills.intersection(freelancer_skills)) / len(required_skills)
    else:
        skill_overlap = 1.0
        
    weighted_skill = skill_overlap * 0.3
    
    normalized_rating = (rating / 5.0) * 0.2
    normalized_active = (active_days / 7.0) * 0.1
    
    final_score = weighted_project + weighted_skill + normalized_rating + normalized_active
    return round(final_score, 4)

# ==========================================
# 5. MAIN ORCHESTRATOR
# ==========================================
def execute_smart_match(project: Dict[str, Any], all_freelancers: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    current_date = datetime.now(timezone.utc)
    scored_candidates = []
    
    for freelancer in all_freelancers:
        if not passes_hard_constraints(freelancer, project, current_date):
            continue
            
        score = calculate_match_score(freelancer, project)
        
        scored_candidates.append({
            "freelancer_id": freelancer["user_id"],
            "match_score": score,
            "bio": freelancer["bio"]
        })
        
    scored_candidates.sort(key=lambda x: x["match_score"], reverse=True)
    return scored_candidates[:5]

In [41]:
execute_smart_match(project,data)

LLM Scoring Failed: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 26.956849859s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'qu

[{'freelancer_id': 'c8a4b632-9c1f-4d9a-8e2b-7f1a3d5e8c90',
  'match_score': 0.7214,
  'bio': 'Senior Backend Architect with deep expertise in Relational Database Management Systems, query optimization, and complex state management.'},
 {'freelancer_id': '9a8b7c6d-5e4f-3a2b-1c0d-9e8f7a6b5c4d',
  'match_score': 0.2514,
  'bio': 'Systems-level programmer with a strong foundation in Operating Systems, memory management, and high-performance C++ applications.'},
 {'freelancer_id': 'f47ac10b-58cc-4372-a567-0e02b2c3d479',
  'match_score': 0.2314,
  'bio': 'Specialist in Large Language Model architecture and Agentic AI workflows. Experienced in fine-tuning models and building robust RAG pipelines.'},
 {'freelancer_id': '2b3c4d5e-6f7a-8b9c-0d1e-2f3a4b5c6d7e',
  'match_score': 0.2314,
  'bio': 'Full-Stack developer bridging the gap between elegant UI and robust API design. Highly ranked in competitive programming circles.'},
 {'freelancer_id': '3c4d5e6f-7a8b-9c0d-1e2f-3a4b5c6d7e8f',
  'match_sco

In [42]:
project

{'project_id': 'proj_a1b2c3d4',
 'client_id': 'client_98765',
 'title': 'Database Normalization and PostgreSQL Migration',
 'description': 'We need a database architect to redesign our legacy inventory schema. The current structure has multiple BCNF violations causing update anomalies. You will be responsible for normalizing the tables to 3NF/BCNF while ensuring dependency preservation, and migrating the existing data to a new PostgreSQL instance.',
 'required_skills': ['PostgreSQL',
  'Database Normalization',
  'Data Migration',
  'SQL'],
 'budget_range': {'min': 1500, 'max': 3000},
 'deadline': '2026-06-30T23:59:59Z',
 'project_type': 'Fixed',
 'status': 'Open',
 'created_at': '2026-05-20T15:17:31Z'}